<a href="https://colab.research.google.com/github/JuanAcevedo08/DeepLearning/blob/main/initorduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parte 1 (FLujo).

## 1. Instalacion de kaggle y TensorFlow

Kaggle para obtener los datos y que el modelo funcione

In [ ]:
url_data_set = 'https://www.kaggle.com/datasets/blastchar/telco-customer-churn'

In [ ]:
#instalar liubreria kaggle
%%capture
!pip install kaggle

In [ ]:
#Acceder a los datasets mediante la api
from google.colab import files
files.upload()

In [ ]:
%%capture
#Mover el archivo creado a la rua que kaggle pide
#Crea un directorio
!mkdir -p ~/.kaggle
#Crea una copia del archivo json cargado antes y lo mete a la ruta
!cp kaggle.json ~/.kaggle/
#Habilida el archivo
!chmod 600 /root/.kaggle/kaggle.json

Ahora ya se pueden mirar los datasets desde colab

## 2. Conectar al dataset

In [ ]:
!kaggle datasets list -s customer-churn

In [ ]:
#Descargar el dataset que queremos
!kaggle datasets download -d blastchar/telco-customer-churn

In [ ]:
!unzip /content/telco-customer-churn.zip #descompirmir el archivo

Normalmente estos casos se pueden resolver en machine learning pero si la tarea es muy compleja y no está teniendo buena solucion entonces optamos por usar deeplearning

## 3. Estructuración de un proyecto de deep Learning


Una vez descargado conectado el dataset listo par trabajar seguimos con los pasos que se tienen que hacer par aun proyecto de deep learning , parecidos a los de machine learning , exploracion, limpieza, preparacion, analisis, entrenamiento, validacion produccion

### Preparación de los datos:


- Definir las columnas relevantes

- Cuales van a apartorar

- Analisis exploratorio de las variables

### Entrenamiento:

- Separar en partes los datos para train/test

### Red Neuronal:

- Creación de la red neuronal

### Validacion del modelo con metricas

- Revisar rendimiento del modelo

### Guardar el modelo:

- Guardar la estrucutra y sus pesos

### Producción

- Se utiliza el modelo guardado para ponerlo en producción a realizar inferencias

# Parte 2 (Creación).

## 0. Instalar dependencias

In [ ]:
#Generar reportes
%%capture
!pip install ydata-profiling

In [ ]:
#Importar librerias a utilizar
import joblib as jb #Guardar el modelo
import pandas as pd #Manipular tablass de datos
import matplotlib.pyplot as plt #Manipular graficos
import seaborn as sns #Crear graficos

import tensorflow as tf #Librería de creación de NN
from tensorflow import keras #Api para manejar la manipulacion de redes neuronales mas facil
from tensorflow.keras.models import load_model #Cargar una red neuronal guardada de tf
from tensorflow.keras.callbacks import EarlyStopping #Callback para parado automatico

from sklearn.preprocessing import LabelEncoder, MinMaxScaler#Preprocesadores para variables categoricas y numericas
from sklearn.model_selection import train_test_split #Separamiento de los datos
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc #Metricas de validacion del modelo
from sklearn.metrics import ConfusionMatrixDisplay

## 1. Preparación Datos (Exploracion, Analisis, eliminacion de variables, Revision Nulos, Normalizacion)

### 1. Exploracion

In [ ]:
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv', sep=',') #Cargar los datos en la ruta que se encuentra
df_copy = df.copy() #Crear una copia para dejar el original limpio
df_copy.head(5) #revisar primeros 5 ejemplos

In [ ]:
pd.set_option('display.max_columns', None) #configuracion para desplegar todas las columnas

In [ ]:
df_copy.head(2) #ver los dos primros ejemplos competos para revisar las variables que tenemos

In [ ]:
df_copy.shape

### 2. EDA (Exploration Data Analyst)

Si queremos revisar valroes unicos , mirar insights en las variables realizamos este procedimineto

In [ ]:
#Funcion para realizar este analisis

def eda(df_):
  #Desscripcion general  transpuesto
  summary = df_.describe(include='all').T
  summary['Type'] = df_.dtypes #Tipo de datos
  summary['Unique Values'] = df_.nunique() #Cantidad de valores unicos
  summary['Examples'] = df_.apply(lambda x: x.dropna().unique()[:3]) #Ejemplo de 3 valores sin nulos

  #reorganziar columnass para visualziacion
  summarry = summary[['Type', 'Unique Values', 'Examples']]
  return summary

In [ ]:
eda(df_copy)

In [ ]:
#Realizar un reporte mas interactiva
from ydata_profiling import ProfileReport
ProfileReport(df_copy, minimal=True)

### 3) Eliminacion Columnas Ruidosas(Eliminar columnas que no aportan informacion relevante)


- CustomerId: Esta variables es un identificador no aporta informacion real por lo que causa ruido

- Gender al tener un desbalanceo de esto , esto puede tener un bias o sesgo para el modelo

In [ ]:
df_copy = df_copy.drop('customerID', axis=1)
df_copy.head(1)

In [ ]:
df_copy = df_copy.drop('gender', axis=1)
df_copy.head(1)

### 4) Analisis y Eliminacion de valores nulos

In [ ]:
#Revisar si tenemos valores nulos
df_copy.isna().sum()

En la priemra parte no contamos con valores nulos , sin embargo estos se tratan mediante df[columna] = df[columna].fillna(df[columna].media, .mediana, .moda) o si son pocos valores un df = df.dropna()

Mas adelante con el trataminto de la variable que dice categorica de totalchargues pero en realidad es numerica vamos a encontrar nulos

O como alternativa debido a que totalchargues tenia valores nulos o categrocios que hacen su comprotamiento al aplicar un label este se encarga de trasnformar esa columna y los nulos o categroicos pasan a ser 0

### 5) Duplicados (revision y eliminacion)

In [ ]:
df_copy.duplicated().sum()

In [ ]:
df_copy[df_copy.duplicated()].head(5)

In [ ]:
df_copy.drop_duplicates(inplace=True)
df_copy.duplicated().sum()

De esa manera evitamos que haya informaacion muy parecida o igual para que el modello no memorice

### 6) Normalizar Columnas Categrocias

Esto es para las columnas categoricas que tienen valores categoricos binarios poder pasarlo a valores numericos binarios que es lo que en este caso necesitamos para poder entrenar el modelo

In [ ]:
#Evitar aletars por uso de replace
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
categorical_col = list(df_copy.select_dtypes(include='O').keys())
# categorical_col = df_copy.select_dtypes(include='object').columns.to_list()
for col in categorical_col:
  df_copy[col] = df_copy[col].replace('Yes', 1)
  df_copy[col] = df_copy[col].replace('No', 0)

In [ ]:
categorical_col

In [ ]:
df_copy.head(1)

In [ ]:
#Aplicar label enconder para estas columnas las cuales no son binarias
labels_encoders = {} #Guardar los cneonder de cada columna debido a que el label enconder funciona solo para columna x columna

for col in categorical_col:
  enconder = LabelEncoder()
  df_copy[col] = enconder.fit_transform(df_copy[col].astype(str)) #Trasnformar y asegurar que sea strign lo que esté transformando
  labels_encoders[col] = enconder #Guardar el enconder de la columna

In [ ]:
#Guardar los codificadores
jb.dump(labels_encoders, 'labels_encoders.pkl')
print('Codificadores Guardados')

In [ ]:
df_copy.head(2)

### 7) Normalizacion variables numericas

In [ ]:
numerical_num = ['MonthlyCharges', 'TotalCharges', 'tenure']
#Debiod a que la mayoria de los datos están entre 0 y 1 utilizamos un minmaxscaler par que l modelo no se conufnda
scaler = MinMaxScaler()
df_copy[numerical_num] = scaler.fit_transform(df_copy[numerical_num])

In [ ]:
jb.dump(scaler, 'scaler.pkl')
print('Escalador fue guardado con exito')

se reliza la guardada de los escaladores y labels encoders ya que esto permite ayuda luego a haacer inferencias con nuevos datos pasen por esos mismos que fueron ajustados previamente y sea mass facil y el modelo no se confunda

## 2) Separación de datos

### 1) Separar variables (Features y Target)

In [ ]:
X = df_copy.drop('Churn', axis=1)
Y = df_copy['Churn']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=42, test_size=0.2, stratify=Y)

#Tamapaño de entrenamiento
print(f'XTrain: {X_train.shape} YTrain: {Y_train.shape}')
print(f'XTest: {X_test.shape} YTest: {Y_test.shape}')

## 3) Red Neuronal

### Red Neuronal Normal

In [ ]:
#Nuemero de columnas (Tamaño de las entradas , siempre al comenzar red neuronal se sepecifica el tamaño de las entradas)
input_shape = 18

In [ ]:
regularizador = tf.keras.regularizers

In [ ]:
#Red neuronal
model_ = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_shape,)), #Capa de entrada con la cantidad de columnas que se tienen
    tf.keras.layers.Dense(input_shape, activation='relu', kernel_regularizer=regularizador.l1(0.01)),
    tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=regularizador.l2(0.01)), #Capa oculta - > reduciendo la cantidad de neuronas en la capa oculta
    tf.keras.layers.Dense(10, activation='relu', kernel_regularizer=regularizador.l1_l2(0.01)),
    tf.keras.layers.Dense(1, activation='sigmoid') #Capa de salida
])

#Revisar estructura del modelo
model_.summary()

In [ ]:
#Visualizar estructura visualmente
keras.utils.plot_model(model_, show_shapes=True)

In [ ]:
model_.compile(optimizer='Adam',loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
early_s_call = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model_.fit(X_train, Y_train, epochs=50, validation_data=(X_test, Y_test), callbacks=[early_s_call])

### Red Neuronal Autotuner

#### Instalación

In [ ]:
#Instalar la librería necesaria
%%capture
!pip install keras-tuner

In [ ]:
#Importar el buscador
from keras_tuner import RandomSearch

In [ ]:
#Tamaño de las entradas
input_shape = 18

#### Funcion Creacion

In [ ]:
def build_model_keras_tuner(hp, input_size=input_shape):
  model = tf.keras.Sequential([
      keras.layers.Input(shape=(input_size,))
  ])

  #Para capa 1 ajustar los parametros para esa capa y que random pueda buscarlo
  model.add(keras.layers.Dense(units=hp.Int('units_layer1', min_value=10, max_value=16, step=2), activation=hp.Choice('activation_layer1', values=['relu', 'tanh'])))
  #Para dropout
  model.add(keras.layers.Dropout(rate=hp.Float('dropout_layer1', min_value=0.0, max_value=0.5, step=0.1)))
  #Para capa 2 (Opcional para que el modelo vea si es mejor con o sin esa capa)
  if hp.Boolean('second_layer'):
    model.add(keras.layers.Dense(units=hp.Int('units_layer2', min_value=5, max_value=10, step=1), activation=hp.Choice('activation_layer2', values=['relu', 'tanh'])))
    model.add(keras.layers.Dropout(rate=hp.Float('dropout_layer2', min_value=0.0, max_value=0.5, step=0.1)))

  #Capa de salida normal
  model.add(keras.layers.Dense(1, activation='sigmoid'))

  #Se compila directamente en la funcion para poder elegir el optimizador
  model.compile(
      optimizer=hp.Choice('optimizer', values=['adam', 'adamW']),
      loss='binary_crossentropy',
      metrics=['accuracy']
  )

  return model

#### Configuracion KerasTuner

In [ ]:
tuner = RandomSearch(
    hypermodel=build_model_keras_tuner,#Funcion creada para el optimizador
    objective='val_accuracy', #Metrica a monitorear -- Optimziar
    max_trials=10, #Numero de combinaciones a probar
    executions_per_trial=2, #Numero de ejecuciones por combinacion
    directory='mdl_dir', #Carpeta para guardar los reustlados
    project_name='tuner_project_churn' #Nombre del proyecto
    )

#### Ejecutar y obtener los mejores resultados detectados

In [ ]:
#Ejecutar con 20 epochs
tuner.search(X_train, Y_train, epochs=20, validation_data=(X_test, Y_test))

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print('Mejores hypers encontrados para capa 1 no capa opcional:')
print(f'Neuronas primera capa : {best_hps.get('units_layer1')}')
print(f'Optimizer: {best_hps.get('optimizer')}')
print(f'Dropout primera capa : {best_hps.get('dropout_layer1')}')

#### Construir el modelo con los mejores params

In [ ]:
best_model = tuner.hypermodel.build(best_hps) #Construir

In [ ]:
#Graficarlo
keras.utils.plot_model(best_model, show_shapes=True)

#### Entrenar el mejor modelo ahora

In [ ]:
#Agregar un earlye Sstopping
early_s_call = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [ ]:
history = best_model.fit(X_train, Y_train, epochs=50, validation_data=(X_test, Y_test), callbacks=[early_s_call])

## 4) Metrics

No es suficiente mirar la disminucion de la perdida para saber si esta está o no optima , para eso entrnan las maetricas

### Confussion Matrix

In [ ]:
pred = (best_model.predict(X_test) > 0.5).astype(int)
cm = confusion_matrix(Y_test, pred)
cm_dataframe = pd.DataFrame(cm)

In [ ]:
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm)
cm_display.plot(cmap='Blues') # Changed colormap to Blues

### Report (Include = [ Accuracy, F1, Recall, Specificity])

In [ ]:
g_report = classification_report(Y_test, pred, target_names=['No', 'Yes'])
print(g_report)

### Graficacion accuracy y loss x Epochs

In [ ]:
def plot_history(history_, metric):
  if metric in ['accuracy', 'loss']:
    plt.figure(figsize=(9, 6))
    if metric == 'accuracy':
      plt.plot(history_.history['accuracy'], 'g-x', label='Accuracy Train')
      plt.plot(history_.history['val_accuracy'], 'r-x', label='Accuracy Test')
    else:
      plt.plot(history_.history['loss'], 'g-x', label='Loss Train')
      plt.plot(history_.history['val_loss'], 'r-x', label='Loss Test')
  else:
    print('Metrica invalida | Metricas validas: [accuracy, loss]')

  plt.title(f'Graficación de {metric}', color='purple', weight='bold')
  plt.xlabel('Epohcs', color='black', weight='bold')
  plt.ylabel(f'{metric}', color='black', weight='bold')
  plt.legend()
  plt.tight_layout()
  plt.show()

In [ ]:
plot_history(history, 'accuracy')

In [ ]:
plot_history(history, 'loss')

### Curva bajo la roca

In [ ]:
pred_train = (best_model.predict(X_train) > 0.5)
#Calcular metricas de la curva para train
fpr_train, tpr_train, _, = roc_curve(Y_train, pred_train)
roc_auc_train = auc(fpr_train, tpr_train) #Area bajo la curva

#Calcular metricas de la curva para test
fpr_test, tpr_test, _ = roc_curve(Y_test, pred)
roc_auc_test = auc(fpr_test, tpr_test) #Area bajo la curva

#Graficar
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, color='blue', lw=2, label=f'Train ROC(Auc = {roc_auc_train:.2f})')
plt.plot(fpr_test, tpr_test, color='green', lw=2, label=f'Valdidation ROC(Auc = {roc_auc_test:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Random Guess')
plt.xlim([0.0, 1.0])
plt.xlim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14, color='black', weight='bold')
plt.ylabel('True Positive Rate', fontsize=14, color='black', weight='bold')
plt.title('Receiver Operating Characteristics (ROC)', fontsize=14, color='black', weight='bold')
plt.legend(loc='lower right', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

## 5) Guardado y cargado modelos

In [ ]:
#Guardar el modelo
path_dir = 'best_model_train.keras' #Ruta
best_model.save(path_dir)

## 6) Inferencias

In [ ]:
import joblib as jb
import pandas as pd

from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [ ]:
#Cargar los preprocesadores de informacion
labels_enconder_inference = jb.load('/content/labels_encoders.pkl')
scaler_inference = jb.load('/content/scaler.pkl')

#Cargar el modelo
model_inference = load_model('/content/best_model_train.keras')

In [ ]:
labels_enconder_inference

In [ ]:
def preprocess_new_data(df_):
  #Se realizan los mismos preprocesaodres de datos que se hicieron para hacer infrencias con nuevos datos de la empresa
  df_ = df_.drop(['customerID', 'gender'], axis=1)

  #remplazamos datos de si y no para quellos valores que sean binarios
  col_cat = list(df_.select_dtypes(include='O').keys())
  for col in col_cat:
    df_[col] = df_[col].replace('Yes', 1)
    df_[col] = df_[col].replace('No', 0)

  #Aplicar label enconder a las columnas
  for col , key in labels_enconder_inference.items():
    #Asegurar que la columna del dataframe ingresada esté en la de los decodificadores
    if col in df_.columns:
      #Asegurar de que a parte de que la columna esté dentro de los enconders también sea tipo str
      df_[col] = df_[col].astype(str).str.strip()
      try:
        df_[col] = key.transform(df_[col])
      except ValueError as e:
        raise ValueError(
            f'Error al transformar la columna {col} '
            f'Asegurate de que los valores en los nuevos datos coincidan con los datos de entrenamiento '
            f'More Details {e}'
        )
  #Aplicar escalado a los valores numericos de el nuevo df
  scale_cols = ['MonthlyCharges', 'TotalCharges', 'tenure']
  df_[scale_cols] = scaler_inference.transform(df_[scale_cols])

  return df_

In [ ]:
def make_inference(new_data, model):
  new_data = pd.DataFrame(new_data) #Se convierte tipo dataframe el nuevo dato

  processed_new_data = preprocess_new_data(new_data) #Se procesa el nuevo dato

  #En caso de que el nuevo dato tenga la columna churn
  if 'Churn' in list(processed_new_data.columns):
    processed_new_data = processed_new_data.drop('Churn', axis=1, errors='ignore')

  predict = model.predict(processed_new_data) #Realizar la prediccion
  score = predict[0][0] * 100
  print(f'Probabilidad de que abandone {score:.2f}%')

  prediction_binary = (predict > 0.5).astype(int)
  print('Churn: ', prediction_binary[0][0])

In [ ]:
#nuevo ejemplo
jun_ejemplo = {
    "customerID": ["9237-HQITU"],
    "gender": ["Female"],
    "SeniorCitizen": [0],
    "Partner": ["No"],
    "Dependents": ["No"],
    "tenure": [2],
    "PhoneService": ["Yes"],
    "MultipleLines": ["No"],
    "InternetService": ["Fiber optic"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["No"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["No"],
    "StreamingMovies": ["No"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [70.70],
    "TotalCharges": [151.65],
}

In [ ]:
make_inference(jun_ejemplo, model_inference)